# Cubic Hermite Bending Beam

Linear Lagrange elements can represent straight edges only. A single cubic-Hermite line element carries both a position *and* a tangent at each node, so it can represent a smoothly bending curve with one element — something no amount of linear elements will ever do exactly.

This notebook builds a 1-D Hermite line mesh, visualizes the four cubic Hermite basis functions, and compares the reconstructed curve to the linear interpolant.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import pyfieldml as fml
from pyfieldml.builders.mesh import add_hermite_mesh
from pyfieldml.model.region import Region

## Build a bending beam

Two nodes at x=(0,0) and x=(1,0) with tangents chosen to pull the curve up in the middle. Tangent magnitude controls how aggressively the curve bends between the end-points.

In [ ]:
r = Region(name="bending_beam")
nodes = np.array([[0.0, 0.0], [1.0, 0.0]], dtype=np.float64)
derivs = np.array([[1.0, 2.0], [1.0, -2.0]], dtype=np.float64)
elements = np.array([[1, 2]], dtype=np.int64)
add_hermite_mesh(
    r,
    name="beam",
    nodes=nodes,
    elements=elements,
    derivatives=derivs,
    topology="line",
)
doc = fml.Document.from_region(r)
print("evaluators:", list(doc.evaluators))

## The four cubic Hermite basis functions

On the reference line xi in [0, 1] the cubic Hermite basis is:

- H0(xi) = 1 - 3xi^2 + 2xi^3  (value at node 1)
- H1(xi) = xi - 2xi^2 + xi^3  (slope at node 1)
- H2(xi) = 3xi^2 - 2xi^3      (value at node 2)
- H3(xi) = -xi^2 + xi^3       (slope at node 2)

In [ ]:
xi = np.linspace(0.0, 1.0, 101)
H0 = 1 - 3 * xi**2 + 2 * xi**3
H1 = xi - 2 * xi**2 + xi**3
H2 = 3 * xi**2 - 2 * xi**3
H3 = -(xi**2) + xi**3

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(xi, H0, label="H0 (value @ node 1)")
ax.plot(xi, H1, label="H1 (slope @ node 1)")
ax.plot(xi, H2, label="H2 (value @ node 2)")
ax.plot(xi, H3, label="H3 (slope @ node 2)")
ax.set_xlabel("xi")
ax.set_ylabel("H_i(xi)")
ax.set_title("Cubic Hermite basis functions on [0, 1]")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
fig.tight_layout()

## Evaluate the beam geometry

`Field.evaluate` assembles the tangent-aware DOFs and returns a point for each xi. The y-coordinate is non-zero in the interior despite both end-points being at y=0 — that's the derivative DOFs at work.

In [ ]:
coords = doc.field("coordinates")
xi_samples = np.linspace(0.0, 1.0, 50).reshape(-1, 1)
elems = np.ones(50, dtype=np.int64)
curve = coords.evaluate(element=elems, xi=xi_samples)
print("sample points on the beam:")
print(curve[::10])

## Compare against the linear interpolant

A linear Lagrange element through the same two nodes is a straight segment; the Hermite element picks up the imposed tangents and bends.

In [ ]:
linear_y = np.zeros_like(xi_samples[:, 0])  # straight line y=0

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(curve[:, 0], linear_y, "--", label="linear Lagrange (straight)")
ax.plot(curve[:, 0], curve[:, 1], "-", lw=2, label="cubic Hermite (bending)")
ax.scatter(nodes[:, 0], nodes[:, 1], color="k", zorder=5, label="nodes")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Single element: Hermite bends, Lagrange cannot")
ax.legend(fontsize=8)
ax.set_aspect("equal")
ax.grid(alpha=0.3)
fig.tight_layout()

### Takeaway

Cubic Hermite is the reason CMISS-family cardiac/musculoskeletal models can reproduce smooth anatomical surfaces with a handful of elements. `add_hermite_mesh` also supports `quad` (bicubic) and `hex` (tricubic) topologies — see the builder docstring for the DOF slot conventions.